# 3.1 Data split 40/10/10/40 (run in SageMaker Studio)

Reads the offline feature store via Athena and assigns **chronological** splits within each subject+activity (train 40% / validation 10% / test 10% / production 40%, with a 1-window gap dropped between blocks). Test and production contain subjects seen in training, so scores are optimistic relative to leave-one-subject-out evaluation.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3
from src.pamap2.config import load_config, s3_uri
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
import io, time, pandas as pd
athena = session.client("athena")
s3 = session.client("s3")
def run(sql, timeout=600, database=None):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=cfg["athena_workgroup"],
        QueryExecutionContext={"Database": database or cfg["glue_database"]})["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        qe = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        st = qe["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"Athena query exceeded {timeout}s: {sql[:200]}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql}")
    if qe.get("StatementType") != "DML":
        return None
    out = qe["ResultConfiguration"]["OutputLocation"]
    b, k = out[len("s3://"):].split("/", 1)
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
import json
from src.pamap2 import features as F
from src.pamap2.split import (assign_splits, split_shares, SPLITS, restrict_to_train_classes,
                              class_coverage)
FG_NAME = cfg["feature_group"]
sm = session.client("sagemaker")
dc = sm.describe_feature_group(FeatureGroupName=FG_NAME)["OfflineStoreConfig"]["DataCatalogConfig"]
tbl = f'"{dc["Database"]}"."{dc["TableName"]}"'
print("Offline table:", tbl)

## Load latest record per `window_id`

In [ ]:
sql = f"""
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY window_id ORDER BY event_time DESC, write_time DESC) AS rn
  FROM {tbl}
) WHERE rn = 1 AND is_deleted = false
"""
df = run(sql, database=dc["Database"]).drop(columns=["rn"])
meta = ["window_id", "subject_id", "session", "start_idx", "activity_id", "intensity", "event_time", "t_start"]
df = df[meta + F.FEATURE_COLUMNS]
print(df.shape, "windows:", df.window_id.nunique())

## Assign splits and report shares

In [ ]:
out = assign_splits(df)
print(f"Dropped {len(df) - len(out)} gap windows; kept {len(out)}")
print("Overall share (%):"); print(split_shares(out))
print("Per activity:"); display(split_shares(out, "activity_id"))
print("Per subject:"); display(split_shares(out, "subject_id"))
shares = split_shares(out)
for k, t in zip(SPLITS, (40, 10, 10, 40)):
    assert abs(shares[k] - t) <= 5, f"{k} share {shares[k]} outside +/-5 of {t}"
assert out.window_id.is_unique

out, dropped, classes = restrict_to_train_classes(out)
print("Classes in train:", classes)
print("Dropped (class absent from train), per split:", dropped.to_dict() or "none")
print("Class coverage (windows per activity x split):"); display(class_coverage(out))

## Write split files to S3

In [ ]:
import pathlib
label_map = {int(a): i for i, a in enumerate(classes)}   # train classes only; XGBoost needs 0..K-1
out = out.assign(label=out.activity_id.map(label_map)).sort_values("window_id").reset_index(drop=True)
cols = list(F.FEATURE_COLUMNS)
tmp = pathlib.Path("/tmp/pamap2_splits"); tmp.mkdir(parents=True, exist_ok=True)
def put(path, key):
    s3.upload_file(str(path), cfg["bucket"], f"{cfg['s3_prefixes']['splits']}{key}")
for name in SPLITS:
    part = out[out.split == name].reset_index(drop=True)
    p = tmp / f"{name}.parquet"; part[meta + ["label", "split"] + cols].to_parquet(p, index=False); put(p, p.name)
    if name != "production":
        c = tmp / f"{name}.csv"                                  # label first, no header
        part[["label"] + cols].to_csv(c, header=False, index=False); put(c, c.name)
    print(name, len(part))
(tmp / "feature_columns.json").write_text(json.dumps(cols)); put(tmp / "feature_columns.json", "feature_columns.json")
(tmp / "label_map.json").write_text(json.dumps([{"activity_id": a, "label": l} for a, l in label_map.items()])); put(tmp / "label_map.json", "label_map.json")
print("Written to", s3_uri(cfg, "splits"))